# EXP-101 · 저신뢰 문항만 Qwen3-VL(A100=8B, T4=4B)로 재추론 (선지 순열 × 멀티크롭)

**근거**: EXP-007 valid 분석 결과 margin(1·2위 확률 차) < 0.9인 78문항(12%)에 오답 33개 중 24개가 몰려 있음. 나머지 594문항은 정답률 98.5%.

**방법**
1. 기존 최고 앙상블(EXP-007×0.52 + EXP-034×0.48)의 margin 계산
2. margin < `MARGIN_TH` 문항만 선택 (valid 약 80개, test 약 800개)
3. Qwen3-VL-8B로 재추론: **선지 순서 4가지(순환 이동)** × **이미지(원본 + 4분할 크롭)** → 확률 평균
4. 기존 확률과 가중 결합 → valid로 가중치 결정 → test 적용

**GPU**: A100이 있으면 이 노트북에 쓰세요(8B bf16 원본, 약 20~40분). T4는 자동으로 4B fp16 사용 (4bit 양자화는 T4에서 30배 느려서 제외).

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes
!apt-get install -y -qq p7zip-full > /dev/null
import transformers, torch
print("transformers", transformers.__version__, "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os, glob, subprocess, time
import pandas as pd, numpy as np
from google.colab import drive
drive.mount("/content/drive")

SHARE_DIR = "/content/drive/MyDrive/ssafy/ai-challenge"
EXP_DIR, SUB_DIR = f"{SHARE_DIR}/experiment_data", f"{SHARE_DIR}/submissions"
ZIP_PATH = glob.glob(f"{SHARE_DIR}/ssafy-16-2-ai*.zip")[0]
DATA_DIR = "/content/data"
if not glob.glob(f"{DATA_DIR}/**/test.csv", recursive=True):
    os.makedirs(DATA_DIR, exist_ok=True)
    subprocess.run(["7z","x","-y","-pocrisnotenough",f"-o{DATA_DIR}",ZIP_PATH],check=True,stdout=subprocess.DEVNULL)
DATA_ROOT = os.path.dirname(glob.glob(f"{DATA_DIR}/**/test.csv", recursive=True)[0])
resolve = lambda p: p if os.path.isabs(str(p)) else os.path.join(DATA_ROOT, str(p))

P = ["prob_a","prob_b","prob_c","prob_d"]
def probs(path): return pd.read_csv(path).set_index("id")[P]

valid_df = pd.read_csv(f"{EXP_DIR}/EXP-007_train6042_resolution1280_valid_seed43.csv")[["id","path","question","a","b","c","d","answer"]]
test_df  = pd.read_csv(f"{DATA_ROOT}/test.csv")
for d in (valid_df, test_df): d["path"] = d["path"].map(resolve)

# 기존 최고 앙상블 (EXP-040 조합)
v_base = 0.52*probs(f"{EXP_DIR}/EXP-007_train6042_resolution1280_valid_seed43.csv").loc[valid_df.id].values \
       + 0.48*probs(f"{EXP_DIR}/EXP-034_qwen7b_language_only_train1600_resolution1280_valid_seed43.csv").loc[valid_df.id].values
t_base = 0.52*probs(f"{EXP_DIR}/EXP-007_train6042_resolution1280_test_probabilities_seed43.csv").loc[test_df.id].values \
       + 0.48*probs(f"{EXP_DIR}/EXP-040_qwen7b_train1600_resolution1280_test_probabilities_seed43.csv").loc[test_df.id].values
def margin(p): s=np.sort(p,1); return s[:,-1]-s[:,-2]
ans = valid_df.answer.map({"a":0,"b":1,"c":2,"d":3}).values
v_ok = v_base.argmax(1)==ans
print("기존 앙상블 valid:", v_ok.mean())
for th in [0.25,0.5,0.9,0.99]:
    m = margin(v_base)<th
    print(f"margin<{th}: valid {m.sum()}문항 (오답 {(~v_ok[m]).sum()}/{(~v_ok).sum()}) | test {(margin(t_base)<th).sum()}문항")

In [ ]:
MARGIN_TH = 0.9      # 위 표를 보고 조정 (오답을 대부분 포함하면서 문항 수가 적은 값)
USE_CROPS = True     # False면 원본 이미지 × 순열 4개만 (4배 빠름)

v_sel = valid_df[margin(v_base)<MARGIN_TH].reset_index(drop=True)
t_sel = test_df[margin(t_base)<MARGIN_TH].reset_index(drop=True)
print("재추론 대상 → valid:", len(v_sel), " test:", len(t_sel))

In [ ]:
# ===== 모델 로드 =====
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from PIL import Image
import torch.nn.functional as F
Image.MAX_IMAGE_PIXELS = None

gpu = torch.cuda.get_device_name(0)
BIG_GPU = any(x in gpu for x in ["A100","H100","L4"])
# T4에서 8B+4bit는 문항당 30초 이상(실측) → T4는 4B fp16(1.3초/문항, 실측)
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct" if BIG_GPU else "Qwen/Qwen3-VL-4B-Instruct"
TAG = f"EXP-101_{MODEL_ID.split('/')[-1].lower()}_selective_m{int(MARGIN_TH*100):03d}" + ("_crop" if USE_CROPS else "")
MAX_PIXELS, MIN_PIXELS = 1280*32*32, 256*32*32

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.bfloat16 if BIG_GPU else torch.float16,
                                                    device_map="cuda", attn_implementation="sdpa")
model.eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)
ip = processor.image_processor
try: ip.size = {"shortest_edge": MIN_PIXELS, "longest_edge": MAX_PIXELS}
except Exception as e: print(e)
for k,v in [("min_pixels",MIN_PIXELS),("max_pixels",MAX_PIXELS)]:
    if hasattr(ip,k): setattr(ip,k,v)

tok = processor.tokenizer
def ids_for(ch):
    s = set()
    for x in [ch, ch.upper(), " "+ch, " "+ch.upper()]:
        t = tok.encode(x, add_special_tokens=False)
        if len(t)==1: s.add(t[0])
    return sorted(s)
CHOICE_IDS = [ids_for(c) for c in "abcd"]
print(CHOICE_IDS)

In [ ]:
# ===== 크롭 · 순열 · 채점 =====
def fit(img):
    w,h = img.size
    if w*h > MAX_PIXELS:
        s=(MAX_PIXELS/(w*h))**0.5; img=img.resize((max(32,int(w*s)),max(32,int(h*s))), Image.BICUBIC)
    return img

def crops4(img, ov=0.6):
    """4분할 크롭 (각 크롭이 가로·세로 60%씩 → 가운데가 겹쳐서 글자가 잘려도 다른 크롭에 온전히 포함)"""
    w,h = img.size; cw,ch = int(w*ov), int(h*ov)
    return [img.crop((x,y,x+cw,y+ch)) for x,y in [(0,0),(w-cw,0),(0,h-ch),(w-cw,h-ch)]]

def prompt(q, opts):
    return (f"이미지를 자세히 보고 질문에 답하세요. 이미지 속 글자를 정확히 읽는 것이 중요합니다.\n\n"
            f"질문: {q}\n" + "".join(f"{l}. {o}\n" for l,o in zip("abcd",opts)) +
            "\n정답 선지의 알파벳(a, b, c, d) 한 글자만 출력하세요.")

@torch.inference_mode()
def score(img, q, opts):
    msgs=[{"role":"user","content":[{"type":"image","image":img},{"type":"text","text":prompt(q,opts)}]}]
    text=processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp=processor(text=[text], images=[img], return_tensors="pt").to(model.device)
    try: lg=model(**inp, logits_to_keep=1, use_cache=False).logits[0,-1].float()
    except TypeError: lg=model(**inp, use_cache=False).logits[0,-1].float()
    s=torch.stack([torch.logsumexp(lg[ids],0) for ids in CHOICE_IDS])
    return F.softmax(s,0).cpu().numpy()

def score_perm(img, r):
    """선지 순환 이동 4가지로 물어본 뒤 원래 순서로 되돌려 평균 → 위치 편향 제거"""
    orig=[r["a"],r["b"],r["c"],r["d"]]; acc=np.zeros(4)
    for k in range(4):
        order=[(j+k)%4 for j in range(4)]            # 표시 위치 j에 원래 선지 order[j]
        p=score(img, r["question"], [orig[i] for i in order])
        for j,i in enumerate(order): acc[i]+=p[j]
    return acc/4

def score_item(r):
    img=Image.open(r["path"]).convert("RGB")
    full=score_perm(fit(img), r)
    if not USE_CROPS: return full, full
    cp=np.mean([score_perm(fit(c), r) for c in crops4(img)],0)
    return full, cp

def run(df, out_csv, every=10):
    cols=["id"]+[f"full_{c}" for c in "abcd"]+[f"crop_{c}" for c in "abcd"]
    done=pd.read_csv(out_csv) if os.path.exists(out_csv) else pd.DataFrame(columns=cols)
    todo=df[~df.id.astype(str).isin(set(done.id.astype(str)))]
    print(f"완료 {len(done)} / 남음 {len(todo)}"); rows=[]; t0=time.time()
    for n,(_,r) in enumerate(todo.iterrows(),1):
        try: f,c=score_item(r)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); f=c=np.full(4,.25); print("OOM", r.id)
        rows.append([r.id,*f,*c])
        if n%every==0 or n==len(todo):
            done=pd.concat([done,pd.DataFrame(rows,columns=cols)],ignore_index=True); done.to_csv(out_csv,index=False); rows=[]
            el=time.time()-t0; print(f"{n}/{len(todo)} {el/n:.1f}s/문항, 남은 {el/n*(len(todo)-n)/60:.0f}분")
    return done.set_index("id")

r=v_sel.iloc[0]; t=time.time(); print(r.answer, [x.round(3) for x in score_item(r)], f"{time.time()-t:.1f}s")

## valid 검증 → 가중치 결정

In [ ]:
vr = run(v_sel, f"{EXP_DIR}/{TAG}_valid.csv")
FULL=[f"full_{c}" for c in "abcd"]; CROP=[f"crop_{c}" for c in "abcd"]
idx = margin(v_base)<MARGIN_TH
sub_ans = ans[idx]; base_sel = v_base[idx]
vf, vc = vr.loc[v_sel.id, FULL].values, vr.loc[v_sel.id, CROP].values
print("재추론 대상", idx.sum(), "문항 중 정답 수")
print("  기존 앙상블 :", (base_sel.argmax(1)==sub_ans).sum())
print("  8B 원본×순열:", (vf.argmax(1)==sub_ans).sum())
print("  8B 크롭×순열:", (vc.argmax(1)==sub_ans).sum())

rows=[]
for wc in np.arange(0,0.51,0.1):            # 8B 내부: 원본 vs 크롭
    new = (1-wc)*vf + wc*vc
    for wn in np.arange(0,1.01,0.1):        # 기존 vs 8B
        mix = (1-wn)*base_sel + wn*new
        full_pred = v_base.argmax(1).copy(); full_pred[idx] = mix.argmax(1)
        rows.append((round(wc,1), round(wn,1), (full_pred==ans).sum(), (full_pred==ans).mean()))
grid = pd.DataFrame(rows, columns=["w_crop","w_new","correct","valid_acc"]).sort_values("valid_acc", ascending=False)
print(f"\n기존 valid: {v_ok.sum()}/{len(ans)} = {v_ok.mean():.4f}")
print(grid.head(15).to_string(index=False))

## test 재추론 (끊기면 이 셀만 다시 실행)

In [ ]:
tr = run(t_sel, f"{EXP_DIR}/{TAG}_test.csv")

In [ ]:
W_CROP, W_NEW = 0.3, 0.6     # ← valid 표 상위권 중 '주변 값도 비슷하게 좋은' 조합으로
tidx = margin(t_base)<MARGIN_TH
new_t = (1-W_CROP)*tr.loc[t_sel.id, FULL].values + W_CROP*tr.loc[t_sel.id, CROP].values
final = t_base.copy(); final[tidx] = (1-W_NEW)*t_base[tidx] + W_NEW*new_t

sub = pd.DataFrame({"id": test_df.id, "answer": ["abcd"[i] for i in final.argmax(1)]})
assert sub.id.astype(str).tolist() == pd.read_csv(f"{DATA_ROOT}/sample_submission.csv").id.astype(str).tolist()
out = f"{SUB_DIR}/{TAG}_wc{int(W_CROP*10)}_wn{int(W_NEW*10)}.csv"
sub.to_csv(out, index=False)
old = pd.read_csv(f"{SUB_DIR}/EXP-040_exp007052_exp034048_resolution1280_seed43.csv")
print("저장:", out, "| EXP-040 대비 바뀐 문항:", (sub.answer.values!=old.answer.values).sum())